In [ ]:
import pandas as pd
import numpy as np

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split

In [ ]:
df = pd.read_csv('MC_RDA_dataset.csv')

In [4]:
print("Original shape:", df.shape)

Original shape: (342, 1828)


In [5]:
print(df["target"].value_counts())

target
0    239
1    103
Name: count, dtype: int64


In [6]:
nan_threshold = 0.5

In [7]:
# === Convert Mordred error strings to NaN ===
error_patterns = [
    "module networkx has no attribute",
    "max() arg is an empty sequence",
    "min() arg is an empty sequence",
    "float division by zero"
]

def replace_errors(val):
    if isinstance(val, str):
        for p in error_patterns:
            if p in val:
                return np.nan
    return val

df = df.applymap(replace_errors)

# === Convert numeric columns properly ===
df = df.apply(pd.to_numeric, errors="ignore")

# === Drop columns with too many NaNs ===
nan_fraction = df.isna().mean()
cols_to_drop = nan_fraction[nan_fraction > nan_threshold].index

print(f"Dropping {len(cols_to_drop)} columns with >{nan_threshold*100:.0f}% NaN")

df_clean = df.drop(columns=cols_to_drop)

/tmp/ipykernel_4031/372879389.py:16: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(replace_errors)
/tmp/ipykernel_4031/372879389.py:19: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  df = df.apply(pd.to_numeric, errors="ignore")


Dropping 169 columns with >50% NaN


In [8]:
target = df_clean["target"]
ids = df_clean["id"]

X = df_clean.drop(columns=["target", "id"])
y = target

In [9]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [10]:
class ZeroVarianceFilter(BaseEstimator, TransformerMixin):
    def __init__(self, threshold=1e-8):
        self.threshold = threshold

    def fit(self, X, y=None):
        X = pd.DataFrame(X)
        self.columns_ = X.columns[X.var() > self.threshold]
        return self

    def transform(self, X):
        X = pd.DataFrame(X)
        return X[self.columns_]

In [11]:
class CorrelationFilter(BaseEstimator, TransformerMixin):
    def __init__(self, threshold=0.95):
        self.threshold = threshold

    def fit(self, X, y=None):
        if isinstance(X, pd.DataFrame):
            self.feature_names_in_ = X.columns
        else:
            self.feature_names_in_ = [f"f{i}" for i in range(X.shape[1])]
            X = pd.DataFrame(X, columns=self.feature_names_in_)

        corr = X.corr().abs()
        upper = corr.where(
            np.triu(np.ones(corr.shape), k=1).astype(bool)
        )

        self.to_drop_ = [
            col for col in upper.columns
            if any(upper[col] > self.threshold)
        ]

        self.keep_cols_ = X.columns.difference(self.to_drop_)
        return self

    def transform(self, X):
        X = pd.DataFrame(X, columns=self.feature_names_in_)
        return X[self.keep_cols_]

In [12]:
preprocess = Pipeline([
    ("variance", ZeroVarianceFilter()),
    ("corr", CorrelationFilter(threshold=0.9))
])

In [13]:
preprocess.fit(X_train)

Pipeline(steps=[('variance', ZeroVarianceFilter()),
                ('corr', CorrelationFilter(threshold=0.9))])

In [14]:
X_train_filtered = preprocess.transform(X_train)
X_test_filtered = preprocess.transform(X_test)

In [15]:
final_features = X_train_filtered.columns

print("Final feature count:", len(final_features))

Final feature count: 99


In [ ]:
X_train_filtered.to_csv("X_train_final.csv", index=False)
X_test_filtered.to_csv("X_test_final.csv", index=False)

y_train.to_csv("y_train.csv", index=False)
y_test.to_csv("y_test.csv", index=False)